# Lab 3: Cleaning I (Missing, Wrong, Duplicated)

**DSA 405 · Week 3**

| | |
|---|---|
| **In class** | Friday, Sep 4 |
| **A3 due** | Thursday, Sep 10, 11:59 PM |
| **File** | `wolfpack_dining_raw.csv` |
| **Time** | ~25 min in class, ~55 min at home |

## Overview

In Lab 2 we profiled the dining file and found columns we could not trust. This week
we act on those findings and record every action we take. Every cleaning decision
removes some information; the cleaning log states exactly what was removed, with
counts.

The technical core of the week is one idea in three forms: a value can be present in
the data, look reasonable, and still be wrong.

In [1]:
# ---------------------------------------------------------------------------
# DSA 405 setup
# ---------------------------------------------------------------------------
import pandas as pd, numpy as np, requests, io

DATA = "https://raw.githubusercontent.com/jon-holt/DSA-405-Student/main/datasets/"
# DATA = "data/raw/"          # local users


def load(filename, kind="csv", **kw):
    """Read a class file whether DATA is a URL or a local folder."""
    path = DATA + filename
    if kind == "csv":
        return pd.read_csv(path, **kw)
    if kind == "excel":
        return pd.read_excel(path, **kw)
    if kind == "text":
        return requests.get(path, timeout=30).text if path.startswith("http") else open(path).read()
    if kind == "json":
        if path.startswith("http"):
            return requests.get(path, timeout=30).json()
        import json as _j
        return _j.load(open(path))
    raise ValueError(kind)


pd.set_option("display.width", 160)
print("pandas", pd.__version__)

pandas 3.0.5


---
# Part 1: Explore (in class)

## Task 1.1: The sentinel census, and the two zeros

A **sentinel** is a special value written inside a data column to signal "no real value here," instead of leaving the cell empty.

Read everything as strings first (Lab 2 showed why). Then count the values in the
`score` and `seats` columns:

In [2]:
dining = load("wolfpack_dining_raw.csv", dtype=str)

print("--- score ---")
print(dining.score.str.strip().value_counts(dropna=False).head(8))
print()
print("--- seats ---")
print(dining.seats.str.strip().value_counts(dropna=False).head(8))

--- score ---
score
100.0    14
89.8     11
92.0      9
NaN       8
0         7
88.5      7
87.8*     6
96.3      6
Name: count, dtype: int64

--- seats ---
seats
8       23
6       20
2       20
7       19
1       15
0       14
-999    13
4       12
Name: count, dtype: int64


Two findings:

- `seats` contains **13** values of `-999` and **14** zeros
- `score` contains **7** zeros

The digit `0` appears in both columns. In one column, zero is a real, possible value;
in the other, zero is impossible, so it must mean "not recorded." Deciding which
column is which, with evidence, is Checkpoint question 1. Think about what each column
measures: a food truck really does have zero seats.

## Task 1.2: Type repair

`score` does not convert to numbers cleanly: some values have extra whitespace, some
end with a `*` (this file uses `*` to mark revised scores), and some are the word
`unknown`. We repair the column in steps and check each step:

In [3]:
s = dining.score.str.strip().str.rstrip("*")
score = pd.to_numeric(s, errors="coerce")

print("failed to convert (now NaN):", score.isna().sum())
print("min:", score.min(), "| max:", score.max(), "| mean:", round(score.mean(), 1))

failed to convert (now NaN): 17
min: 0.0 | max: 100.0 | mean: 90.6


Look at the minimum: `min = 0.0`. The conversion succeeded, and that success is the
problem. The seven zero-scores converted without any error, because `0` is a valid
number. `errors="coerce"` turns text that cannot be read as a number into `NaN`, the value
pandas uses to mean "missing." It cannot detect a valid number that is not a real
measurement. We have to write that
check ourselves:

In [4]:
score_clean = score.replace(0, np.nan)

print(f"zeros converted to NaN: {(score == 0).sum()}")
print("min is now:", score_clean.min(), "| mean:", round(score_clean.mean(), 1))

zeros converted to NaN: 7
min is now: 81.5 | mean: 92.4


## Task 1.3: Exact duplicates

Some rows appear in the file twice, identical in every column:

In [5]:
n_dup = dining.duplicated().sum()
print(f"exact duplicate rows: {n_dup}")

deduped = dining.drop_duplicates()
print(f"{len(dining)} rows in, {len(deduped)} out, {len(dining) - len(deduped)} dropped as exact duplicates")

exact duplicate rows: 46
366 rows in, 320 out, 46 dropped as exact duplicates


### Seeing the duplicates, not just counting them

`.duplicated()` marks the *second and later* copies of each repeated row, which is
what makes the count correct. To **look** at the duplicates you want every copy,
including the first one, so pass `keep=False`:

In [6]:
# Every row that belongs to an exact-duplicate group, sorted so the copies appear together
dup_rows = dining[dining.duplicated(keep=False)].sort_values("location_name")

print(f"rows involved in exact duplicates: {len(dup_rows)}")
print(f"distinct rows that got repeated:   {len(dup_rows.drop_duplicates())}")
dup_rows


rows involved in exact duplicates: 74
distinct rows that got repeated:   28


,unit_code,location_name,category,campus_zone,inspection_date,score,seats,avg_ticket,open_now,manager_notes
242,5428,Case Snack Shop,Food truck,Central,2024-07-13,87.2,2,$15.29,Y,Reinspection scheduled
154,5428,Case Snack Shop,Food truck,Central,2024-07-13,87.2,2,$15.29,Y,Reinspection scheduled
316,1249,Jordan Cafe,Food Truck,Centennial,24-May-2024,89.8,3,$9.53,N,Follow up re: hood cleaning
307,1249,Jordan Cafe,Food Truck,Centennial,24-May-2024,89.8,3,$9.53,N,Follow up re: hood cleaning
32,2103,CafÃ© Polk Food Court,FoodTruck,Vet School,05/15/2025,89.8,1,$15.80,yes,NaN
...,...,...,...,...,...,...,...,...,...,...
243,8312,Weaver Pizza Window,food truck,Centennial,2024-02-22 16:15:00,86.8,unknown,$16.80,1,No issues noted
166,4956,Winston Snack Shop,FoodTruck,North,"September 15, 2024",88.0,4,$24.59,yes,Reinspection scheduled
133,4956,Winston Snack Shop,FoodTruck,North,"September 15, 2024",88.0,4,$24.59,yes,Reinspection scheduled
348,4956,Winston Snack Shop,FoodTruck,North,"September 15, 2024",88.0,4,$24.59,yes,Reinspection scheduled


Now look at the same file with the extra copies removed, and ask the important
question: does leaving the duplicates in change the number you would report?

In [7]:
no_dups = dining.drop_duplicates()          # first copy of each group kept


def numeric_score(df):
    """score as a number, with the zeros that mean 'not recorded' removed (Task 1.2)."""
    s = df.score.str.strip().str.rstrip("*")
    return pd.to_numeric(s, errors="coerce").replace(0, np.nan)


with_dups = numeric_score(dining)
without_dups = numeric_score(no_dups)

print(f"with duplicates:    n={with_dups.notna().sum():4d}   mean score = {with_dups.mean():.4f}")
print(f"without duplicates: n={without_dups.notna().sum():4d}   mean score = {without_dups.mean():.4f}")
print(f"shift caused by the duplicates: {with_dups.mean() - without_dups.mean():+.4f}")
print()
print(f"mean score of the duplicated rows themselves: {numeric_score(dup_rows).mean():.4f}")


with duplicates:    n= 342   mean score = 92.4395
without duplicates: n= 296   mean score = 92.7524
shift caused by the duplicates: -0.3129

mean score of the duplicated rows themselves: 90.6027


The duplicated rows are not a random sample of the file: their own mean score is lower
than the mean of the whole file, so every extra copy moves the overall mean down.
Duplicates do not only increase the row count `n`; they also give extra weight to
whichever rows happened to be copied. That is why the count alone is not the full
answer to Checkpoint question 2.

---
## Checkpoint: submit before leaving class

1. The two zeros: in which column is `0` a real, possible value, in which column does
   it mean "not recorded," and what is your evidence?
2. How many exact duplicate rows are in the file, and how would keeping them change
   the mean score?
3. What should `-999` in `seats` become, and why?

*Answers here.*

---
# Part 2: A3 (Cleaning I)

This part is graded. It has four tasks. Task 2.4 counts the most in the rubric.

## Task 2.1: Your sentinel policy

Decide what to do with every sentinel found in `score` and `seats`: the zeros, the
`-999` values, and the text values. This set of decisions is your **sentinel policy**.
For each sentinel: convert it, keep it, or flag it, with one sentence of justification
per decision. "Zero seats is a real value for food trucks" is a justification;
"converted everything to NaN" is not.

In [8]:
# your conversions

*Decisions and justifications here.*

## Task 2.2: A ranking that reverses

Compute the mean `score` by category twice, once treating the zero-scores as real
values and once treating them as missing, and compare the two rankings.

The categories first need consolidating (27 strings, ~6 real categories, profiled in
Lab 2). You will learn to build that mapping properly in Week 4, so this week we provide it
for you:

In [9]:
def canon(s):
    """A simple category canonicalizer: it maps each variant spelling to one standard name. In Week 4 you learn to build this yourself."""
    if pd.isna(s):
        return s
    s = " ".join(s.strip().lower().replace("-", " ").split())
    return {"c store": "convenience", "convenience store": "convenience",
            "coffee shop": "coffee", "café": "cafe",
            "fastcasual": "fast casual", "foodtruck": "food truck"}.get(s, s)

dining["category_clean"] = dining.category.map(canon)
print(dining.category_clean.value_counts())

category_clean
food truck     107
coffee          57
fast casual     53
cafe            51
dining hall     50
convenience     48
Name: count, dtype: int64


In [10]:
# your two group-bys: mean score by category_clean, with and without the zero-scores

Report the two means for the category whose mean changes the most, and report both
rankings. State which category ranks worst under each treatment. Then add two
sentences on how a decision made in Week 3 (your sentinel policy) changes a conclusion
that would be presented in Week 14.

## Task 2.3: Duplicates that the exact check misses

`drop_duplicates()` with no arguments only finds rows that match exactly, character
for character. `"Port City Java "` and `"port city java"` are the same place written
with different case and whitespace.

1. Normalize `location_name` (strip whitespace from the ends, lowercase, collapse
   internal whitespace) into a new column.
2. Count duplicates on `(unit_code, normalized name, inspection_date)`.
3. Report three counts: exact duplicates, duplicates after normalizing, and how many
   the exact check missed. Show two rows that only the normalized check finds.

In [11]:
# your near-duplicate search

## Task 2.4: Start the cleaning log

Collect every cleaning action you took in A3 into a cleaning log: a markdown table
with one row per action.

| # | Action | Rows affected | Rows remaining | Why |
|---|---|---|---|---|

Requirements:

- **8 to 12 rows**, covering every change made (conversions, NaN decisions, drops)
- Every row includes a **count**, and the arithmetic must balance exactly: the rows
  you started with, minus the rows you dropped, must equal the rows you ended with. If
  the numbers do not balance, one of your actions is missing from the log; find it.
- The **Why** column is one specific clause. "Impossible value for this measurement"
  is specific enough; "cleaning" is not.

This table is the starting point for the P2 cleaning log (due Oct 1), where the same
standard applies to your own project data.

*Log here.*

---
## AI use note

Tell me which AI tools you used here and what you used them for, in a sentence or two.
If you didn't use any, write "none."

*Answer here.*

---
## Submitting

1. **Runtime > Restart runtime**, then **Run all**.
2. `File > Download > Download .ipynb`
3. Rename to `DSA405_002_FA26_A3_[yourUnityID].ipynb`
4. Upload to the **A3** space on Moodle.

The **Checkpoint** section is submitted separately to **Week 3 In-Class Activity**, before
the end of class on Friday. Due for A3: **Thursday, Sep 10, 11:59 PM**.